# Using a Voting Regressor to Predict the Severity of Parkinson's Disease from Patients' Voices

In this notebook, we apply a voting regressor to explore if we can classify a predict whether or not a forst fire occurred. This [Parkinsons Telemonitoring dataset](https://archive.ics.uci.edu/dataset/189/parkinsons+telemonitoring) comes from UC Irvine's Machine Learning Repository.

Parkinson's disease is a neurological disorder affecting motor control. One of its hallmarks is a condition called dysarthria, which is a deterioration of speech caused by muscle rigidity and tremor. Researchers at Oxford University and Intel have shown that voice biomarkers extracted from recordings can track disease severity remotely, enabling patients to be monitored from home.

This dataset contains 5,875 voice recordings from 42 patients, collected over six months. Each recording yields 16 biomedical voice features, and two clinician-assigned severity scores, the motor UPDRS and total UPDRS, serve as regression targets. We predict the motor UPDRS (Unified Parkinson's Disease Rating Scale, motor subscale), a continuous score from 0 to 108 measuring motor symptom severity.

**Features (16 inputs):** Jitter(%), Jitter(Abs), Jitter:RAP, Jitter:PPQ5, Jitter:DDP, Shimmer, Shimmer(dB), Shimmer:APQ3, Shimmer:APQ5, Shimmer:APQ11, Shimmer:DDA, NHR, HNR, RPDE, DFA, PPE

**Target:** motor_UPDRS — motor severity score (continuous, range 0–108)

It is recommended to review each algorithm beforehand to get an understanding of how they work. This notebook will just be a simple example showcasing the use of ensemble models.

---

## Load and Explore Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ml_package import (
    LinearRegression,
    KNN,
    DecisionTreeRegressor,
    RandomForestRegressor,
    VotingRegressor,
    StandardScaler,
    train_test_split,
    regression_metrics,
)

np.random.seed(42)

In [8]:
# Load dataset and assign features and targets
df = pd.read_csv("parkinsons_updrs.data")
X = df.drop(columns = ["subject#", "age", "sex", "test_time", "total_UPDRS", "motor_UPDRS"])
y = df["motor_UPDRS"]

In [11]:
# Descriptive statistics
X.describe().round(3)


,Jitter(%),Jitter(Abs),Jitter:RAP,Jitter:PPQ5,Jitter:DDP,Shimmer,Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,Shimmer:APQ11,Shimmer:DDA,NHR,HNR,RPDE,DFA,PPE
count,5875.000,5875.0,5875.000,5875.000,5875.000,5875.000,5875.000,5875.000,5875.000,5875.000,5875.000,5875.000,5875.000,5875.000,5875.000,5875.000
mean,0.006,0.0,0.003,0.003,0.009,0.034,0.311,0.017,0.020,0.027,0.051,0.032,21.679,0.541,0.653,0.220
std,0.006,0.0,0.003,0.004,0.009,0.026,0.230,0.013,0.017,0.020,0.040,0.060,4.291,0.101,0.071,0.091
min,0.001,0.0,0.000,0.000,0.001,0.003,0.026,0.002,0.002,0.002,0.005,0.000,1.659,0.151,0.514,0.022
25%,0.004,0.0,0.002,0.002,0.005,0.019,0.175,0.009,0.011,0.016,0.028,0.011,19.406,0.470,0.596,0.156
50%,0.005,0.0,0.002,0.002,0.007,0.028,0.253,0.014,0.016,0.023,0.041,0.018,21.920,0.542,0.644,0.206
75%,0.007,0.0,0.003,0.003,0.010,0.040,0.365,0.021,0.024,0.033,0.062,0.031,24.444,0.614,0.711,0.264
max,0.100,0.0,0.058,0.070,0.173,0.269,2.107,0.163,0.167,0.275,0.488,0.748,37.875,0.966,0.866,0.732


## Preprocessing

We apply a standard 80/20 train-test split. `StandardScaler` is fitted on training data and applied to all splits, as it is crucial for KNN (distance-based) and linear regression (gradient scale), and harmless but consistent for tree-based models.

---

In [14]:
X_arr = X.values.astype(float)
X_train, X_test, y_train, y_test = train_test_split(X_arr, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")


Train: 4700 samples  |  Test: 1175 samples
Features: 16


## Train Individual Models

In [17]:
# Linear Regression 
lr = LinearRegression()
lr.train(X_train_s, y_train, eta=0.05, epochs=3000)




In [18]:
# KNN Regressor (k=7) 
knn = KNN(k=7, regression=True)
knn.fit(X_train_s, y_train)




In [19]:
# Decision Tree Regressor
dtr = DecisionTreeRegressor(max_depth=7)
dtr.fit(X_train, y_train) 

## Voting Regressor with Equal Weights

We will now aggregate each fitted model by using `VotingRegressor` with equal weights. Weighted-averages are also supported by the model. `VotingRegressor` expects pre-fitted models. We pass raw vs scaled models as needed by wrapping each model's predict to use the right feature space. The simplest solution is to scale the decision tree predictions separately and pass the scaled X to all. Since tree-based models are scale-invariant, scaling them doesn't change predictions. Thus, we can safely pass the test data to all four models.

Also note, I excluded the random forest from this example because of the time it takes to train.

---

In [22]:

dtr_s = DecisionTreeRegressor(max_depth=7)
dtr_s.fit(X_train_s, y_train)


voting_eq = VotingRegressor(models=[
    ("lr",  lr),
    ("knn", knn),
    ("dtr", dtr_s),
])

y_pred_eq = voting_eq.predict(X_test_s)
r2_eq   = regression_metrics.r_squared(y_test, y_pred_eq)
rmse_eq = regression_metrics.rmse(y_test, y_pred_eq)
print(f"VotingRegressor (equal weights)  →  R²: {r2_eq:.4f}   RMSE: {rmse_eq:.4f}")


VotingRegressor (equal weights)  →  R²: 0.2975   RMSE: 6.8608


In [33]:
lr_pred = lr.predict(X_test_s)
lr_r2 = regression_metrics.r_squared(y_test, lr_pred)
lr_r2

np.float64(0.09695950456277602)

In [35]:
knn_pred = knn.predict(X_test_s)
knn_r2 = regression_metrics.r_squared(y_test, knn_pred)
knn_r2

np.float64(0.32018916184331825)

In [37]:
dtr_pred = dtr.predict(X_test_s)
dtr_r2 = regression_metrics.r_squared(y_test, dtr_pred)
dtr_r2

np.float64(-0.3778004890477913)

---

The equal weighted-voting regressor produced a pretty low R-squared value of only about 0.3. However, when compared to the individual R-squared scores, we can see that using an ensemble model to aggregate predictions across models can be more reliable than choosing just one model fit.

---